# Credit Factor Hierarchy

**Purpose:** End-to-end walkthrough of the credit factor hierarchy feature.

**Prerequisites:** Familiarity with bonds, P&L attribution, and spread curves.

**In this notebook:**
1. Build synthetic spread history for 6 issuers (24 months).
2. Define issuer tags and a two-level (Rating, Region) hierarchy.
3. Calibrate a `CreditFactorModel`.
4. Save the artifact to JSON; reload it.
5. Decompose a two-date period with `decompose_levels` and `decompose_period`.
6. Run attribution on a small synthetic portfolio.
7. Build a credit vol forecast report.

In [1]:
import json
import math
import tempfile
from datetime import date as dt_date
from pathlib import Path

import numpy as np

import sys
sys.path.insert(0, "..")

from _shared.instrument_fixtures import fixed_bond
from finstack_quant.attribution import attribute_pnl_from_spec
from finstack_quant.core.dates import ScheduleBuilder
from finstack_quant.core.market_data import DiscountCurve, MarketContext
from finstack_quant.factor_model.credit import (
    CreditCalibrator,
    CreditFactorModel,
    FactorCovarianceForecast,
    LevelsAtDate,
    PeriodDecomposition,
    decompose_levels,
    decompose_period,
)

print("Imports OK.")


Imports OK.


## Step 1 — Synthetic spread history

We generate 24 months of spread data for 6 issuers: 3 IG (EU, NA, APAC) and
3 HY (EU, NA, APAC).  Each issuer's spread is a linear function of a generic
factor plus a small idiosyncratic noise term.  All random numbers use fixed
seeds to ensure deterministic CI execution.

**Panel dates come from `core.dates.ScheduleBuilder`,** not from repeated
`timedelta(days=30)` arithmetic. A fixed 30-day step is not a month: it drifts
roughly five days per year, so a panel labelled "monthly" quickly stops landing
on month ends and can skip a calendar month entirely. `ScheduleBuilder` with a
`1M` frequency and `end_of_month(True)` produces true month-end observation
dates — which is also what makes `2024-02-29` (used as the `T0` snapshot in
Step 5) an actual date in this panel.

In [2]:
rng = np.random.default_rng(42)  # fixed seed — deterministic

N_MONTHS = 24
AS_OF = "2024-03-31"

# Calendar-correct month-end observation dates ending at AS_OF.
# The builder is inclusive of both endpoints, so ask for one extra month and
# keep the trailing N_MONTHS.
_schedule = (
    ScheduleBuilder(dt_date(2022, 3, 31), dt_date.fromisoformat(AS_OF))
    .frequency("1M")
    .end_of_month(True)
    .build()
)
dates = [d.isoformat() for d in _schedule.dates][-N_MONTHS:]
assert len(dates) == N_MONTHS and dates[-1] == AS_OF

# Generic factor: CDX IG 5Y proxy, mean 100 bp with small trend
generic_values = [100.0 + 0.5 * math.sin(i * 0.4) for i in range(N_MONTHS)]

ISSUERS = [
    ("ISSUER-A", "IG", "EU"),
    ("ISSUER-B", "IG", "NA"),
    ("ISSUER-C", "IG", "APAC"),
    ("ISSUER-D", "HY", "EU"),
    ("ISSUER-E", "HY", "NA"),
    ("ISSUER-F", "HY", "APAC"),
]

spreads: dict[str, list] = {}
tags: dict[str, dict] = {}
asof_spreads: dict[str, float] = {}

for k, (issuer_id, rating, region) in enumerate(ISSUERS):
    base = 80.0 + k * 30.0  # IG: ~80-170 bp, HY: ~170-230 bp
    beta_pc = 0.6 + 0.05 * k
    noise = rng.normal(0, 1.5, size=N_MONTHS)
    series = [
        base + beta_pc * (generic_values[i] - 100.0) + noise[i]
        for i in range(N_MONTHS)
    ]
    spreads[issuer_id] = series
    tags[issuer_id] = {"rating": rating, "region": region}
    asof_spreads[issuer_id] = series[-1]

print(f"Dates: {dates[0]} .. {dates[-1]}  ({len(dates)} months, all month-end)")
print(f"Issuers: {list(spreads.keys())}")
print("As-of spreads (bp):")
for k, v in asof_spreads.items():
    print(f"  {k}: {v:.1f}")

Dates: 2022-04-30 .. 2024-03-31  (24 months, all month-end)
Issuers: ['ISSUER-A', 'ISSUER-B', 'ISSUER-C', 'ISSUER-D', 'ISSUER-E', 'ISSUER-F']
As-of spreads (bp):
  ISSUER-A: 79.8
  ISSUER-B: 110.4
  ISSUER-C: 138.4
  ISSUER-D: 167.9
  ISSUER-E: 199.7
  ISSUER-F: 230.5


## Step 2 — Hierarchy spec and calibration config

In [3]:
_NOTEBOOK_DATA = json.loads(Path("data/credit_factor_hierarchy.json").read_text())

calibration_config = _NOTEBOOK_DATA["calibration_config"]

calibration_inputs = {
    "history_panel": {"dates": dates, "spreads": spreads},
    "issuer_tags": {"tags": tags},
    "generic_factor": {
        "spec": {"name": "CDX IG 5Y", "series_id": "cdx.ig.5y"},
        "values": generic_values,
    },
    "as_of": AS_OF,
    "asof_spreads": asof_spreads,
    "idiosyncratic_overrides": {},
}

print("Config hierarchy levels:", calibration_config["hierarchy"]["levels"])
print("Input issuers:", list(calibration_inputs["issuer_tags"]["tags"].keys()))

Config hierarchy levels: ['rating', 'region']
Input issuers: ['ISSUER-A', 'ISSUER-B', 'ISSUER-C', 'ISSUER-D', 'ISSUER-E', 'ISSUER-F']


## Step 3 — Calibrate `CreditFactorModel`

In [4]:
calibrator = CreditCalibrator(json.dumps(calibration_config))
model = calibrator.calibrate(json.dumps(calibration_inputs))

print(f"schema_version : {model.schema_version}")
print(f"as_of          : {model.as_of}")
print(f"n_levels       : {model.n_levels}")
print(f"n_issuers      : {model.n_issuers}")
print(f"n_factors      : {model.n_factors}")
print(f"level_names    : {model.level_names()}")
print(f"issuer_ids     : {model.issuer_ids()}")
print(f"factor_ids     : {model.factor_ids()}")

schema_version : finstack_quant.credit_factor_model/1
as_of          : 2024-03-31
n_levels       : 2
n_issuers      : 6
n_factors      : 9
level_names    : ['Rating', 'Region']
issuer_ids     : ['ISSUER-A', 'ISSUER-B', 'ISSUER-C', 'ISSUER-D', 'ISSUER-E', 'ISSUER-F']
factor_ids     : ['credit::generic', 'credit::level0::Rating::HY', 'credit::level0::Rating::IG', 'credit::level1::Rating.Region::HY.APAC', 'credit::level1::Rating.Region::HY.EU', 'credit::level1::Rating.Region::HY.NA', 'credit::level1::Rating.Region::IG.APAC', 'credit::level1::Rating.Region::IG.EU', 'credit::level1::Rating.Region::IG.NA']


## Step 4 — Save and reload artifact

In [5]:
# Save to a temporary file and reload — verifies the JSON round-trip.
with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False) as f:
    artifact_path = Path(f.name)
    f.write(model.to_json())

artifact_json = artifact_path.read_text()
model_reloaded = CreditFactorModel.from_json(artifact_json)

assert model_reloaded.schema_version == model.schema_version
assert model_reloaded.n_issuers == model.n_issuers
assert model_reloaded.n_factors == model.n_factors
assert model_reloaded.issuer_ids() == model.issuer_ids()

# Clean up
artifact_path.unlink()

print("Artifact saved and reloaded successfully.")
print(f"Reloaded: {model_reloaded}")

Artifact saved and reloaded successfully.
Reloaded: CreditFactorModel(as_of=2024-03-31, n_levels=2, n_issuers=6, n_factors=9)


## Step 5 — Decompose a two-date period

`decompose_levels` computes a snapshot: generic factor value, per-level bucket
values, and per-issuer residual adders.  `decompose_period` diffs two snapshots.

In [6]:
# T0: the second-to-last panel date (2024-02-29). Because the panel is built on
# a calendar-correct month-end schedule, this is a real observation date.
T0_DATE = dates[-2]
T0_GENERIC = 99.5

# T1: as-of (the last panel date, 2024-03-31)
T1_DATE = AS_OF
T1_GENERIC = 101.2
assert T0_DATE == "2024-02-29" and T1_DATE == dates[-1]

spreads_json = json.dumps(asof_spreads)  # same observed spreads at both dates

snap_t0 = decompose_levels(model, spreads_json, T0_GENERIC, T0_DATE)
snap_t1 = decompose_levels(model, spreads_json, T1_GENERIC, T1_DATE)

period = decompose_period(snap_t0, snap_t1)

# decompose_levels -> LevelsAtDate snapshots; decompose_period -> PeriodDecomposition
assert isinstance(snap_t0, LevelsAtDate) and isinstance(period, PeriodDecomposition)

print(f"Period: {period.from_date} → {period.to_date}")
print(f"Δ generic (bp): {period.d_generic:.4f}")

for lvl in range(period.n_levels):
    deltas = period.level_deltas(lvl)
    level_name = model.level_names()[lvl]
    print(f"Level {lvl} ({level_name}) Δ bucket values:")
    for bucket, delta in sorted(deltas.items()):
        print(f"  {bucket}: {delta:+.4f} bp")

d_adder = period.d_adder()
print("Δ adder per issuer (bp):")
for issuer, delta in sorted(d_adder.items()):
    print(f"  {issuer}: {delta:+.4f}")

Period: 2024-02-29 → 2024-03-31
Δ generic (bp): 1.7000
Level 0 (Rating) Δ bucket values:
  HY: -1.7000 bp
  IG: -1.7000 bp
Level 1 (Region) Δ bucket values:
  HY.APAC: +0.0000 bp
  HY.EU: +0.0000 bp
  HY.NA: +0.0000 bp
  IG.APAC: +0.0000 bp
  IG.EU: +0.0000 bp
  IG.NA: +0.0000 bp
Δ adder per issuer (bp):
  ISSUER-A: +0.0000
  ISSUER-B: +0.0000
  ISSUER-C: +0.0000
  ISSUER-D: +0.0000
  ISSUER-E: +0.0000
  ISSUER-F: +0.0000


## Step 6 — Attribution on a small portfolio

We run `attribute_pnl_from_spec` with an `AttributionEnvelope` JSON that
embeds the model inline.  The market context is a **flat discount curve at two
dates one basis point apart** — deliberately hand-built rather than taken from
`_shared.build_demo_market()`, because attribution needs two market snapshots
that differ by a controlled amount, and the shared market is a single snapshot.
There are no hazard curves, so credit P&L is zero in this synthetic example; the
purpose here is to confirm the spec executes end-to-end.

The bond payloads come from `_shared.instrument_fixtures.fixed_bond`, so this
notebook does not re-type a fixed-rate bond wire spec.

In [7]:
def flat_discount_curve(base_date: str, rate: float) -> dict:
    """Build a canonical flat discount market and return its JSON state."""
    context = MarketContext().insert(DiscountCurve.flat("USD-OIS", dt_date.fromisoformat(base_date), rate))
    return json.loads(context.to_json())


AS_OF_T0 = "2025-01-15"
AS_OF_T1 = "2025-01-16"
market_t0 = flat_discount_curve(AS_OF_T0, 0.04)
market_t1 = flat_discount_curve(AS_OF_T1, 0.04 + 0.0001)  # 1 bp shift

print("Market states built.")

Market states built.


In [8]:
def make_bond_spec(idx: int, bond_id: str, coupon: float, years: int) -> dict:
    """Wrap the shared fixed-rate bond fixture for attribution.

    The wire-level payload is `_shared.instrument_fixtures.fixed_bond`; only the
    id, issue date, maturity and coupon are overridden so each bond carries a
    different point on the curve. `fixed_bond` returns a fresh dict per call, so
    mutating it here is safe.
    """
    _, instrument_spec = fixed_bond(idx)
    instrument_spec["spec"]["id"] = bond_id
    instrument_spec["spec"]["issue_date"] = "2025-01-01"
    instrument_spec["spec"]["maturity"] = f"{2025 + years}-01-01"
    instrument_spec["spec"]["cashflow_spec"]["Fixed"]["rate"] = str(coupon)
    return instrument_spec


# Portfolio: 4 bonds with different maturities
PORTFOLIO_BONDS = [
    ("BOND-2Y", 0.045, 2),
    ("BOND-5Y", 0.050, 5),
    ("BOND-7Y", 0.055, 7),
    ("BOND-10Y", 0.060, 10),
]

# Embed the calibrated CreditFactorModel fields directly in the spec.
model_inline = json.loads(model.to_json())

results = []
for idx, (bond_id, coupon, years) in enumerate(PORTFOLIO_BONDS):
    envelope = {
        "schema": "finstack_quant.attribution/1",
        "attribution": {
            "instrument": make_bond_spec(idx, bond_id, coupon, years),
            "market_t0": market_t0,
            "market_t1": market_t1,
            "as_of_t0": AS_OF_T0,
            "as_of_t1": AS_OF_T1,
            "method": "Parallel",
            "config": None,
            "model_params_t0": None,
            "credit_factor_model": model_inline,
            "credit_factor_detail_options": {
                "include_per_bucket_breakdown": True,
                "include_per_issuer_adder": False,
            },
        },
    }
    result_json = attribute_pnl_from_spec(json.dumps(envelope))
    result = json.loads(result_json)
    attr = result["result"]["attribution"]
    results.append({
        "bond": bond_id,
        "total_pnl": float(attr["total_pnl"]["amount"]),
        "carry": float(attr["carry"]["amount"]),
        "rates_curves_pnl": float(attr["rates_curves_pnl"]["amount"]),
        "credit_curves_pnl": float(attr["credit_curves_pnl"]["amount"]),
        "has_credit_detail": attr.get("credit_factor_detail") is not None,
    })

print(f"{'Bond':<10} {'Total P&L':>12} {'Carry':>12} {'Rates':>12} {'Credit':>12} {'CreditDetail?':>14}")
print("-" * 70)
for r in results:
    print(
        f"{r['bond']:<10} {r['total_pnl']:>12.2f} {r['carry']:>12.2f}"
        f" {r['rates_curves_pnl']:>12.2f} {r['credit_curves_pnl']:>12.2f}"
        f" {str(r['has_credit_detail']):>14}"
    )

Bond          Total P&L        Carry        Rates       Credit  CreditDetail?
----------------------------------------------------------------------
BOND-2Y          -81.66       112.14      -193.80         0.00          False
BOND-5Y         -355.19       115.78      -470.98         0.00          False
BOND-7Y         -530.74       120.70      -651.44         0.00          False
BOND-10Y        -787.18       128.50      -915.68         0.00          False


## Step 7 — Credit vol forecast report

`FactorCovarianceForecast` wraps the calibrated model to produce
horizon-scaled covariance matrices and per-issuer idiosyncratic vol estimates.

In [9]:
fcf = FactorCovarianceForecast(model)

# One-step annualized covariance
cov_one = json.loads(fcf.covariance_at("one_step"))
factor_ids = cov_one["factor_ids"]
data = np.array(cov_one["data"]).reshape(len(factor_ids), -1)

print("Factor IDs:")
for fid in factor_ids:
    print(f"  {fid}")

print(f"\nCovariance matrix shape: {data.shape}")
print("Diagonal variances (annualized):")
for fid, var in zip(factor_ids, np.diag(data)):
    print(f"  {fid}: {var:.6f}")

# NSteps(12) covariance = 12 × one-step (Sample vol model)
cov_12 = json.loads(fcf.covariance_at('{"n_steps": 12}'))
data_12 = np.array(cov_12["data"]).reshape(len(factor_ids), -1)
ratio = data_12[0, 0] / data[0, 0] if data[0, 0] != 0.0 else float("nan")
print(f"\nNSteps(12) / OneStep ratio (should be 12.0): {ratio:.4f}")

# Per-issuer idiosyncratic vol
print("\nPer-issuer idiosyncratic vol (one-step, annualized):")
for issuer_id in model.issuer_ids():
    vol = fcf.idiosyncratic_vol(issuer_id, "one_step")
    print(f"  {issuer_id}: {vol:.6f}")

Factor IDs:
  credit::generic
  credit::level0::Rating::HY
  credit::level0::Rating::IG
  credit::level1::Rating.Region::HY.APAC
  credit::level1::Rating.Region::HY.EU
  credit::level1::Rating.Region::HY.NA
  credit::level1::Rating.Region::IG.APAC
  credit::level1::Rating.Region::IG.EU
  credit::level1::Rating.Region::IG.NA

Covariance matrix shape: (9, 9)
Diagonal variances (annualized):
  credit::generic: 0.241281
  credit::level0::Rating::HY: 6.349253
  credit::level0::Rating::IG: 14.032439
  credit::level1::Rating.Region::HY.APAC: 28.822305
  credit::level1::Rating.Region::HY.EU: 18.549063
  credit::level1::Rating.Region::HY.NA: 46.917123
  credit::level1::Rating.Region::IG.APAC: 26.509076
  credit::level1::Rating.Region::IG.EU: 24.306435
  credit::level1::Rating.Region::IG.NA: 17.684772

NSteps(12) / OneStep ratio (should be 12.0): 12.0000

Per-issuer idiosyncratic vol (one-step, annualized):
  ISSUER-A: 0.000000
  ISSUER-B: 0.000000
  ISSUER-C: 0.000000
  ISSUER-D: 0.000000
  ISS

## Takeaways

- `CreditCalibrator` fits a hierarchical spread factor model from a panel of
  issuer spreads and a generic (CDX-like) factor time series.
- `decompose_levels` / `decompose_period` decompose spread moves into generic,
  bucket-level, and issuer-specific (adder) components.
- `attribute_pnl_from_spec` with `credit_factor_model` embedded in the spec
  activates the opt-in credit factor detail in P&L attribution.
- `FactorCovarianceForecast` forecasts factor and idiosyncratic vols at
  arbitrary horizons for risk and scenario analytics.